# RAASTA M2 — Add datasets ONE BY ONE (best effort today)

**Do this order.** After each ADD cell, run **STATS**. At the end run **TRAIN** once (saves Colab time).

| Step | Cell | Dataset |
|------|------|---------|
| 0 | Setup | GPU + Roboflow key + Drive |
| A | ADD A | Roboflow core (bumps + degradation) |
| B | ADD B | Extra Roboflow potholes / bumps |
| C | ADD C | Kaggle `road-anomaly-ds` (YOLO ready) |
| D | ADD D | Kaggle pothole/cracks (optional) |
| E | ADD E | India-like bumps (Roboflow / Mendeley upload) |
| F | ADD F | SciDB / GitHub (optional manual zip) |
| ★ | TRAIN + EXPORT | One strong train, then phone `.tflite` |

**Runtime → T4 GPU** before starting.

If a cell fails: skip it and continue — more data is useless if Colab dies.

In [ ]:
#@title 0) Setup — Roboflow key + install + Drive
ROBOFLOW_API_KEY = ""  #@param {type:"string"}
assert ROBOFLOW_API_KEY.strip(), "Paste Roboflow API key"

!pip -q uninstall -y pillow pillow-simd 2>/dev/null
!pip -q install -U "pillow==11.2.1" "ultralytics>=8.3.0" roboflow opencv-python-headless pyyaml kaggle

from google.colab import drive
drive.mount("/content/drive")

import os, shutil, random, yaml, zipfile
from pathlib import Path
from collections import Counter

import torch
from ultralytics import YOLO
from roboflow import Roboflow

DRIVE_DIR = Path("/content/drive/MyDrive/RAASTA_models")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/raasta_m2_best")
RAW = WORK / "raw"
MERGED = WORK / "merged"
for p in (WORK, RAW):
    p.mkdir(parents=True, exist_ok=True)

CLASSES = ["pothole", "crack", "speed_bump"]
NAME_TO_ID = {n: i for i, n in enumerate(CLASSES)}
ALIASES = {
    "pothole": "pothole", "potholes": "pothole", "hole": "pothole",
    "pot hole": "pothole", "pot-hole": "pothole",
    "crack": "crack", "cracks": "crack", "longitudinal crack": "crack",
    "transverse crack": "crack", "alligator crack": "crack",
    "road degradation": "crack", "degradation": "crack",
    "speed_bump": "speed_bump", "speed-bump": "speed_bump", "speedbump": "speed_bump",
    "speed bump": "speed_bump", "speedbreaker": "speed_bump", "speed breaker": "speed_bump",
    "speed-breaker": "speed_bump", "bump": "speed_bump", "hump": "speed_bump",
    "speed hump": "speed_bump", "unmarked bump": "speed_bump",
}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def canon(name: str):
    n = str(name).strip().lower().replace("_", " ")
    return ALIASES.get(n) or ALIASES.get(n.replace(" ", "_")) or ALIASES.get(str(name).strip().lower())

def ensure_merged():
    for sub in ("images/train", "images/val", "labels/train", "labels/val"):
        (MERGED / sub).mkdir(parents=True, exist_ok=True)
    yml = MERGED / "data.yaml"
    if not yml.is_file():
        yml.write_text(yaml.safe_dump({
            "path": str(MERGED),
            "train": "images/train",
            "val": "images/val",
            "names": {i: n for i, n in enumerate(CLASSES)},
        }))

def load_names(root: Path):
    for fname in ("data.yaml", "dataset.yaml"):
        yml = root / fname
        if yml.is_file():
            data = yaml.safe_load(yml.read_text())
            names = data.get("names")
            if isinstance(names, dict):
                return [names[k] for k in sorted(names, key=lambda x: int(x))]
            if isinstance(names, list):
                return names
    return []

def find_pairs(root: Path):
    items = []
    patterns = [
        ("train", ["images/train", "train/images", "train"]),
        ("val", ["images/val", "images/valid", "val/images", "valid/images", "val", "valid"]),
        ("test", ["images/test", "test/images", "test"]),
    ]
    for split_key, candidates in patterns:
        out_split = "val" if split_key in ("val", "test") else "train"
        for cand in candidates:
            img_dir = root / cand
            if not img_dir.is_dir():
                continue
            lbl_guesses = [
                root / cand.replace("images", "labels"),
                img_dir.parent / "labels",
                root / "labels" / Path(cand).name,
                img_dir,
            ]
            for img in img_dir.rglob("*"):
                if img.suffix.lower() not in IMG_EXTS:
                    continue
                lbl = None
                for lg in lbl_guesses:
                    t = lg / f"{img.stem}.txt"
                    if t.is_file():
                        lbl = t
                        break
                if lbl is None:
                    found = list(root.rglob(f"{img.stem}.txt"))
                    lbl = found[0] if found else None
                items.append((out_split, img, lbl))
            break
    return items

def merge_yolo_root(src: Path, tag: str, full_frame_class=None):
    """Merge a YOLO folder. If full_frame_class set and no labels, label whole image."""
    ensure_merged()
    names = load_names(src)
    id_map = {}
    for i, nm in enumerate(names):
        c = canon(str(nm))
        if c:
            id_map[i] = NAME_TO_ID[c]
    added = 0
    counts = Counter()
    pairs = find_pairs(src)
    if not pairs and full_frame_class is not None:
        # flat image folder → full-frame boxes
        for img in src.rglob("*"):
            if img.suffix.lower() in IMG_EXTS:
                pairs.append(("train" if random.random() > 0.15 else "val", img, None))
    for split, img, lbl in pairs:
        lines_out = []
        if lbl is not None and Path(lbl).is_file():
            for line in Path(lbl).read_text().strip().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                old = int(float(parts[0]))
                if id_map:
                    if old not in id_map:
                        continue
                    new_id = id_map[old]
                elif full_frame_class is not None:
                    new_id = NAME_TO_ID[full_frame_class]
                else:
                    continue
                lines_out.append(" ".join([str(new_id)] + parts[1:5]))
                counts[CLASSES[new_id]] += 1
        elif full_frame_class is not None:
            new_id = NAME_TO_ID[full_frame_class]
            # nearly full image (cropped bump datasets)
            lines_out = [f"{new_id} 0.5 0.5 0.92 0.92"]
            counts[CLASSES[new_id]] += 1
        if not lines_out:
            continue
        stem = f"{tag}_{img.stem}_{added}"
        shutil.copy2(img, MERGED / "images" / split / f"{stem}{img.suffix.lower()}")
        (MERGED / "labels" / split / f"{stem}.txt").write_text("\n".join(lines_out) + "\n")
        added += 1
    print(f"[{tag}] added images={added} boxes={dict(counts)}")
    return added

def download_rf(workspace, project, version, folder_name):
    out = RAW / folder_name
    if out.exists() and any(out.rglob("*.jpg")):
        print("[skip exists]", folder_name)
        return out
    print("[download RF]", workspace, project, version)
    ds = Roboflow(api_key=ROBOFLOW_API_KEY.strip()).workspace(workspace).project(project).version(version).download(
        "yolov8", location=str(out)
    )
    return Path(ds.location)

def stats():
    ensure_merged()
    n_img = len(list((MERGED / "images/train").glob("*"))) + len(list((MERGED / "images/val").glob("*")))
    c = Counter()
    for lbl in (MERGED / "labels").rglob("*.txt"):
        for line in lbl.read_text().splitlines():
            if line.strip():
                c[CLASSES[int(float(line.split()[0]))]] += 1
    print("TOTAL images:", n_img, "| boxes:", dict(c))

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable T4 GPU first"
ensure_merged()
print("Ready. Run ADD A next.")

## ADD A — Roboflow core (must run)
Speed bumps + road degradation + unmarked bumps.

In [ ]:
#@title ADD A) Roboflow core
jobs = [
    ("speed-bump-detection", "speed-bump-detection-se0eh", 15, "speed_bump_v15"),
    ("nsip-project", "road-degradation-beta", 1, "road_degradation"),
    ("pothole-detection-1nczj", "speed-unmarked-bumb", 1, "unmarked_bump"),
]
for ws, proj, ver, name in jobs:
    try:
        root = download_rf(ws, proj, ver, name)
        merge_yolo_root(root, name)
    except Exception as e:
        print("[warn]", name, e)
stats()

## ADD B — Extra Roboflow potholes / bumps
Skip quietly if a project 404s.

In [ ]:
#@title ADD B) Extra Roboflow
jobs = [
    ("brad-dwyer", "pothole-voxrl", 1, "pothole_brad"),
    ("project-jnlc6", "pothole-detection-yolov8", 1, "pothole_yolov8"),
    ("alia-khalifa", "detecting-speed-bumps", 4, "india_like_bumps_rf"),
]
for ws, proj, ver, name in jobs:
    try:
        root = download_rf(ws, proj, ver, name)
        merge_yolo_root(root, name)
    except Exception as e:
        print("[warn skip]", name, e)
stats()

## ADD C — Kaggle heavy YOLO set (recommended today)

1. Kaggle → Settings → API → **Create New Token** → downloads `kaggle.json`
2. Upload it with the cell below (or skip ADD C)

Dataset: [`david2do/road-anomaly-ds`](https://www.kaggle.com/datasets/david2do/road-anomaly-ds)  
Classes already: Pothole, Speedbump, Crack (perfect for RAASTA).

In [ ]:
#@title ADD C) Upload kaggle.json + download road-anomaly-ds
from google.colab import files

kg = Path("/root/.kaggle")
kg.mkdir(parents=True, exist_ok=True)
target = kg / "kaggle.json"
if not target.is_file():
    print("Upload kaggle.json …")
    uploaded = files.upload()
    assert "kaggle.json" in uploaded, "Need kaggle.json"
    target.write_bytes(uploaded["kaggle.json"])
os.chmod(target, 0o600)

out = RAW / "road_anomaly_ds"
if not (out.exists() and any(out.rglob("*.jpg"))):
    !kaggle datasets download -d david2do/road-anomaly-ds -p /content/raasta_m2_best/raw --unzip
    # find actual root
    cands = [p for p in RAW.iterdir() if p.is_dir() and ("anomaly" in p.name.lower() or (p / "data.yaml").exists() or (p / "train").exists())]
    if not out.exists() and cands:
        if out.exists():
            shutil.rmtree(out)
        shutil.move(str(cands[0]), str(out))
    # sometimes unzip flat into RAW
    if not out.exists():
        out.mkdir(parents=True, exist_ok=True)
        for item in list(RAW.iterdir()):
            if item.name in ("road_anomaly_ds",) or item.suffix == ".zip":
                continue
            if item.name in ("train", "val", "test", "data.yaml") or item.suffix in (".yaml", ".yml"):
                shutil.move(str(item), str(out / item.name))

print("Looking under", RAW)
root = out if out.exists() else RAW
# prefer folder that has data.yaml
for p in [out, *RAW.rglob("data.yaml")]:
    if p.name == "data.yaml":
        root = p.parent
        break
    if (p / "data.yaml").is_file():
        root = p
        break
print("Using", root)
merge_yolo_root(root, "kaggle_anomaly")
stats()

## ADD D — Extra Kaggle potholes/cracks (optional)
[`sabidrahman/pothole-cracks-and-openmanhole`](https://www.kaggle.com/datasets/sabidrahman/pothole-cracks-and-openmanhole)  
Needs same `kaggle.json`. Skip if download is huge / fails.

In [ ]:
#@title ADD D) Kaggle pothole-cracks (optional)
try:
    dest = RAW / "kaggle_pothole_cracks"
    if not (dest.exists() and any(dest.rglob("*.jpg"))):
        !kaggle datasets download -d sabidrahman/pothole-cracks-and-openmanhole -p /content/raasta_m2_best/raw/kaggle_pothole_cracks --unzip
    root = dest
    for p in dest.rglob("data.yaml"):
        root = p.parent
        break
    n = merge_yolo_root(root, "kaggle_pc")
    if n == 0:
        # try classification-style folders named pothole/crack
        for folder, cls in (("pothole", "pothole"), ("potholes", "pothole"), ("crack", "crack"), ("cracks", "crack")):
            hits = [p for p in dest.rglob(folder) if p.is_dir()]
            for h in hits[:1]:
                merge_yolo_root(h, f"kaggle_pc_{cls}", full_frame_class=cls)
    stats()
except Exception as e:
    print("[skip ADD D]", e)

## ADD E — Mendeley India speed bumps

Raw Mendeley files are often **cropped photos without YOLO boxes**.

**Easiest today (pick one):**
1. Already covered partly by ADD B `alia-khalifa/detecting-speed-bumps`
2. Or download zip from [Mendeley bvpt9xdjz8](https://data.mendeley.com/datasets/bvpt9xdjz8/1) on your PC → upload zip below  
   → we auto-label each crop as a full-frame `speed_bump`

In [ ]:
#@title ADD E) Upload Mendeley zip (optional) OR skip
from google.colab import files

UPLOAD_MENDELEY = False  #@param {type:"boolean"}
if UPLOAD_MENDELEY:
    print("Upload the Mendeley .zip …")
    up = files.upload()
    zpath = Path(list(up.keys())[0])
    out = RAW / "mendeley_bumps"
    out.mkdir(exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall(out)
    merge_yolo_root(out, "mendeley_india", full_frame_class="speed_bump")
    stats()
else:
    print("Skipped Mendeley upload. ADD B Roboflow bumps already help India-like roads.")

## ADD F — SciDB / GitHub (optional manual)

SciDB is often hard to download automatically from Colab (China mirror / login).

If you have a YOLO zip ready:
1. Set `UPLOAD_EXTRA = True`
2. Upload zip with `images/` + `labels/` or Roboflow layout

Otherwise **skip** — don’t block today’s train.

In [ ]:
#@title ADD F) Upload extra YOLO zip (SciDB/GitHub) — optional
from google.colab import files

UPLOAD_EXTRA = False  #@param {type:"boolean"}
FULL_FRAME_CLASS = ""  #@param {type:"string"}  # leave empty if labels exist; or pothole/crack/speed_bump

if UPLOAD_EXTRA:
    print("Upload YOLO dataset zip …")
    up = files.upload()
    zpath = Path(list(up.keys())[0])
    out = RAW / "extra_upload"
    if out.exists():
        shutil.rmtree(out)
    out.mkdir()
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall(out)
    ff = FULL_FRAME_CLASS.strip() or None
    if ff and ff not in NAME_TO_ID:
        raise ValueError("FULL_FRAME_CLASS must be pothole, crack, or speed_bump")
    merge_yolo_root(out, "extra", full_frame_class=ff)
    stats()
else:
    print("Skipped ADD F")
stats()

## ★ TRAIN once (after A–C at least) + export TFLite

Recommended minimum today: **A + B + C**, then this cell.

If Colab runs out of memory: set `BATCH = 4`.

In [ ]:
#@title TRAIN + EXPORT (run once at the end)
from google.colab import files

stats()
n_train = len(list((MERGED / "images/train").glob("*")))
assert n_train > 300, f"Only {n_train} train images — run ADD A/B/C first"

prev = DRIVE_DIR / "raasta_m2_best.pt"
MODEL = str(prev) if prev.is_file() else "yolov8n.pt"
print("Starting from:", MODEL)

EPOCHS = 80
IMGSZ = 416
BATCH = 8  # change to 4 if OOM

model = YOLO(MODEL)
model.train(
    data=str(MERGED / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,
    project=str(WORK / "runs"),
    name="m2_all_data",
    exist_ok=True,
    patience=20,
    cos_lr=True,
    close_mosaic=15,
    plots=False,
)

best = Path(model.trainer.best)
metrics = model.val(data=str(MERGED / "data.yaml"), imgsz=IMGSZ, batch=BATCH, plots=False)
print("mAP50:", float(metrics.box.map50))
print("Per-class mAP50:", [float(x) for x in metrics.box.ap50])

shutil.copy2(best, DRIVE_DIR / "raasta_m2_best.pt")
exp = YOLO(str(best)).export(format="tflite", imgsz=320)
tflite = Path(exp)
if tflite.suffix != ".tflite":
    cands = list(tflite.parent.glob("*.tflite"))
    tflite = cands[0]
shutil.copy2(tflite, DRIVE_DIR / "raasta_m2.tflite")
local = WORK / "raasta_m2.tflite"
shutil.copy2(tflite, local)
print("Drive:", DRIVE_DIR / "raasta_m2.tflite")
files.download(str(local))
print("Replace assets/models/raasta_m2.tflite on PC → flutter run")

### Today’s checklist

1. Setup (0)
2. ADD A → ADD B → ADD C  ← **do these for sure**
3. ADD D / E / F only if time and Colab is stable
4. TRAIN + EXPORT once
5. Put `raasta_m2.tflite` in `assets/models/` and restart the app

**Target:** overall mAP50 ≥ **0.55** (better than your ~0.41 first model).